# ML1 Task 2 — Developer Salary Prediction
## Feature Engineering v6

This notebook creates leakage-safe feature matrices for model search. It keeps the final corrected preprocessing logic and adds model-search support for the most useful critique points:

1. Training-only target variants: raw log salary, winsorized log salary, floor-500 log salary, and a mask for dropping training rows below 500 USD.
2. Validation and internal-test targets remain raw and unchanged for honest RMSE.
3. `build.vs.buy` is one-hot encoded instead of ordinal encoded.
4. `OLS_SAFE` is stricter: it drops squared terms and drops `experience.years`, keeping `coding.years.professional` as the main experience measure.
5. Region target encoding and Duan smearing are intentionally left for the modeling notebook because they must be handled during cross-validation/evaluation to avoid leakage.


In [1]:

"""
ML1 (2026) Task 2 — Developer Salary Prediction
Feature Engineering v6

Purpose
-------
Create leakage-safe feature matrices for model search with ML1 models:
OLS, Ridge, LASSO, Elastic Net, KNN regression, and SVR variants.

What v6 changes compared with the previous final version
-------------------------------------------------------
1. Keeps the leakage-safe split and train-only preprocessing from the final version.
2. Adds training-target variants for salary-quality experiments:
   - raw log salary
   - train-only winsorized log salary
   - train-only floor-at-500 log salary
   - a mask for dropping training rows below 500 USD if we want to test that variant
3. Does NOT clean validation/internal-test targets; those remain raw for honest RMSE.
4. Moves build.vs.buy from ordinal encoding to one-hot encoding because it is not clearly ordinal.
5. Makes OLS_SAFE stricter:
   - keeps coding.years.professional
   - drops experience.years from OLS_SAFE because of high correlation with professional coding years
   - drops squared numeric terms from OLS_SAFE
   - keeps squared terms in FULL for regularized/nonlinear models
6. Keeps OLS_SAFE multi-select counts; FULL keeps multi-select binaries and non-redundant counts.

Important
---------
This is still the model-search feature engineering file. The final modeling notebook should decide
which target variant and model actually perform best on raw validation RMSE.
"""

from __future__ import annotations

import json
import pickle
import re
import warnings
from collections import Counter
from dataclasses import dataclass, field
from pathlib import Path
from typing import Dict, Iterable, List, Tuple

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

# -----------------------------------------------------------------------------
# Configuration
# -----------------------------------------------------------------------------
TARGET = "annual.pay.usd"
ID_COL = "id"

DEFAULT_SEED = 42
DEFAULT_VAL_SIZE = 0.15
DEFAULT_INTERNAL_TEST_SIZE = 0.15

SPLIT_STRATIFY_BY_REGION = True
SPLIT_REGION_MIN_COUNT = 10

MISSING_FLAG_THRESHOLD = 0.10
RARE_MIN_COUNT_NOMINAL = 20
RARE_MIN_COUNT_REGION = 10

WINSOR_LOWER_Q = 0.01
WINSOR_UPPER_Q = 0.99
WINSOR_MIN_REGION_N = 30

# Low-salary variants are saved only for training experiments.
# Validation/internal-test raw targets are never cleaned.
LOW_SALARY_FLOOR_USD = 500.0

MULTI_SEP = ";"

NUMERIC_FINAL_FULL_COLS = [
    "coding.years.professional",
    "experience.years",
    "job.satisfaction",
    "years.before.professional",
    "coding.years.professional_sq",
    "experience.years_sq",
]

# OLS_SAFE deliberately avoids the most collinear/nonlinear numeric terms.
NUMERIC_FINAL_OLS_SAFE_COLS = [
    "coding.years.professional",
    "job.satisfaction",
    "years.before.professional",
]

NUMERIC_ALL_COLS = list(dict.fromkeys(NUMERIC_FINAL_FULL_COLS + NUMERIC_FINAL_OLS_SAFE_COLS))

ORDINAL_MAPS = {
    "age.group": {"18-24": 0, "25-34": 1, "35-44": 2, "45-54": 3, "55+": 4},
    "education": {
        "Primary/elementary school": 0,
        "Secondary school (e.g. American high school, German Realschule or Gymnasium, etc.)": 1,
        "Some college/university study without earning a degree": 2,
        "Associate degree (A.A., A.S., etc.)": 3,
        "Something else": 3,
        "Bachelor's degree (B.A., B.S., B.Eng., etc.)": 4,
        "Master's degree (M.A., M.S., M.Eng., MBA, etc.)": 5,
        "Professional degree (JD, MD, Ph.D, Ed.D, etc.)": 6,
    },
    "company.size": {
        "Just me - I am a freelancer, sole proprietor, etc.": 0,
        "2 to 9 employees": 1,
        "10 to 19 employees": 2,
        "20 to 99 employees": 3,
        "100 to 499 employees": 4,
        "500 to 999 employees": 5,
        "1,000 to 4,999 employees": 6,
        "5,000 to 9,999 employees": 7,
        "10,000 or more employees": 8,
    },
    "tech.purchase.influence": {
        "I have little or no influence": 0,
        "I have some influence": 1,
        "I have a great deal of influence": 2,
    },
    "ai.sentiment": {
        "Very unfavorable": 0,
        "Unfavorable": 1,
        "Indifferent": 2,
        "Unsure": 2,
        "Favorable": 3,
        "Very favorable": 4,
    },
    "ai.trust": {
        "Highly distrust": 0,
        "Somewhat distrust": 1,
        "Neither trust nor distrust": 2,
        "Somewhat trust": 3,
        "Highly trust": 4,
    },
    "ai.complex.rating": {
        "Very poor at handling complex tasks": 0,
        "Bad at handling complex tasks": 1,
        "Neither good or bad at handling complex tasks": 2,
        "Good, but not great at handling complex tasks": 3,
        "Very well at handling complex tasks": 4,
    },
    "daily.search.time": {
        "Less than 15 minutes a day": 0,
        "15-30 minutes a day": 1,
        "30-60 minutes a day": 2,
        "60-120 minutes a day": 3,
        "Over 120 minutes a day": 4,
    },
    "daily.answer.time": {
        "Less than 15 minutes a day": 0,
        "15-30 minutes a day": 1,
        "30-60 minutes a day": 2,
        "60-120 minutes a day": 3,
        "Over 120 minutes a day": 4,
    },
}

QUASI_ORDINAL_MAPS = {
    "is.dev.professional": {
        "I am a developer by profession": 1,
        "I am not primarily a developer, but I write code sometimes as part of my work/studies": 0,
    },
    "people.manager": {"People manager": 1, "Individual contributor": 0},
    "uses.ai": {"Yes": 2, "No, but I plan to soon": 1, "No, and I don't plan to": 0},
    # build.vs.buy is intentionally NOT here in v6; it is one-hot encoded.
}

ONEHOT_COLS = [
    "region",
    "employment.type",
    "work.location",
    "dev.role",
    "industry",
    "cloud.hosting",
    "first.help.source",
    "ai.job.threat",
    "build.vs.buy",
]

MULTI_SELECT_TOP = {
    "prog.languages": 20,
    "databases": 15,
    "cloud.platforms": 15,
    "web.frameworks": 15,
    "other.tech": 15,
    "dev.tools": 15,
    "dev.environments": 15,
    "personal.os": 10,
    "work.os": 10,
    "project.mgmt.tools": 12,
    "comm.tools": 12,
    "ai.search.tools": 12,
    "ai.tools.used": 12,
    "side.coding": 8,
    "how.learned.coding": 9,
}

CATEGORY_REPLACEMENTS = {
    "dev.role": {"Other (please specify):": "Other", "Other:": "Other"},
    "industry": {"Other:": "Other", "Other (please specify):": "Other"},
    "first.help.source": {"Other:": "Other", "Other (please specify):": "Other"},
}

# -----------------------------------------------------------------------------
# Utilities
# -----------------------------------------------------------------------------
def normalize_text_value(value):
    """Normalize text while preserving missing values."""
    if pd.isna(value):
        return np.nan
    value = str(value).strip()
    value = value.replace("’", "'").replace("‘", "'").replace("`", "'")
    value = re.sub(r"\s+", " ", value)
    return value


def normalize_object_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Normalize all object columns and selected category labels."""
    df = df.copy()
    for col in df.select_dtypes(include=["object"]).columns:
        df[col] = df[col].map(normalize_text_value)
    for col, replacements in CATEGORY_REPLACEMENTS.items():
        if col in df.columns:
            df[col] = df[col].replace(replacements)
    return df


def sanitize_feature_name(name: str) -> str:
    """Create safe feature names for multi-select indicators."""
    name = normalize_text_value(name)
    name = str(name).lower()
    replacements = {
        " ": "_", "/": "_", ".": "_", "(": "", ")": "", ",": "",
        "'": "", "-": "_", "+": "plus", "#": "sharp", ":": "", "!": "",
        "&": "and", "[": "", "]": "",
    }
    for old, new in replacements.items():
        name = name.replace(old, new)
    name = re.sub(r"[^a-z0-9_]+", "", name)
    name = re.sub(r"_+", "_", name).strip("_")
    return name


def parse_multiselect_cell(value, sep: str = MULTI_SEP) -> List[str]:
    """Split one multi-select cell into cleaned individual items."""
    if pd.isna(value):
        return []
    value = normalize_text_value(value)
    if value == "":
        return []
    return [normalize_text_value(item) for item in str(value).split(sep) if normalize_text_value(item) != ""]


def make_region_strata(region_series: pd.Series, min_count: int = SPLIT_REGION_MIN_COUNT) -> pd.Series:
    """Group rare regions for safer stratified splitting."""
    counts = region_series.value_counts(dropna=False)
    return region_series.where(region_series.map(counts) >= min_count, "__RARE_REGION__")


def require_columns(df: pd.DataFrame, required: Iterable[str], df_name: str) -> None:
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"{df_name} is missing required columns: {missing}")


def safe_log_salary(y: pd.Series) -> pd.Series:
    """Log-transform salary, clipping at 1 only to avoid log(0)."""
    return np.log(pd.Series(y).astype(float).clip(lower=1))


# -----------------------------------------------------------------------------
# Target processing
# -----------------------------------------------------------------------------
@dataclass
class TargetProcessor:
    """Creates optional train-only winsorized target variant."""
    lower_q: float = WINSOR_LOWER_Q
    upper_q: float = WINSOR_UPPER_Q
    min_region_n: int = WINSOR_MIN_REGION_N
    target: str = TARGET
    region_col: str = "region"
    region_bounds_: Dict[str, Tuple[float, float]] = field(default_factory=dict)
    global_bounds_: Tuple[float, float] = (np.nan, np.nan)

    def fit(self, df_train: pd.DataFrame) -> "TargetProcessor":
        require_columns(df_train, [self.target, self.region_col], "df_train")
        y = df_train[self.target].astype(float)
        self.global_bounds_ = (float(y.quantile(self.lower_q)), float(y.quantile(self.upper_q)))

        self.region_bounds_ = {}
        region_counts = df_train[self.region_col].value_counts(dropna=False)
        for region, n in region_counts.items():
            vals = df_train.loc[df_train[self.region_col] == region, self.target].astype(float)
            if n >= self.min_region_n:
                bounds = (float(vals.quantile(self.lower_q)), float(vals.quantile(self.upper_q)))
            else:
                bounds = self.global_bounds_
            self.region_bounds_[str(region)] = bounds
        return self

    def winsorize_training_target(self, y: pd.Series, regions: pd.Series) -> pd.Series:
        """Apply train-fitted bounds. Intended only for training target experiments."""
        if not self.region_bounds_:
            raise RuntimeError("TargetProcessor must be fitted first.")
        y_out = pd.Series(y, index=y.index).astype(float).copy()
        regions = pd.Series(regions, index=y.index)
        for region in regions.unique():
            lo, hi = self.region_bounds_.get(str(region), self.global_bounds_)
            mask = regions == region
            y_out.loc[mask] = y_out.loc[mask].clip(lower=lo, upper=hi)
        return y_out


# -----------------------------------------------------------------------------
# Feature processing
# -----------------------------------------------------------------------------
@dataclass
class DeveloperSalaryPreprocessor:
    """Fit-on-train-only feature engineering pipeline."""
    missing_flag_threshold: float = MISSING_FLAG_THRESHOLD
    rare_min_count_nominal: int = RARE_MIN_COUNT_NOMINAL
    rare_min_count_region: int = RARE_MIN_COUNT_REGION
    multi_select_top: Dict[str, int] = field(default_factory=lambda: MULTI_SELECT_TOP.copy())
    onehot_cols: List[str] = field(default_factory=lambda: ONEHOT_COLS.copy())
    ordinal_maps: Dict[str, Dict[str, float]] = field(default_factory=lambda: ORDINAL_MAPS.copy())
    quasi_ordinal_maps: Dict[str, Dict[str, float]] = field(default_factory=lambda: QUASI_ORDINAL_MAPS.copy())

    missing_indicator_cols_: List[str] = field(default_factory=list)
    numeric_medians_: Dict[str, float] = field(default_factory=dict)
    ordinal_fallbacks_: Dict[str, float] = field(default_factory=dict)
    rare_keep_values_: Dict[str, List[str]] = field(default_factory=dict)
    multi_top_items_: Dict[str, List[str]] = field(default_factory=dict)
    multi_unique_item_counts_: Dict[str, int] = field(default_factory=dict)
    multi_include_count_full_: Dict[str, bool] = field(default_factory=dict)

    feature_cols_full_: List[str] = field(default_factory=list)
    feature_cols_ols_safe_: List[str] = field(default_factory=list)
    zero_variance_full_: List[str] = field(default_factory=list)
    zero_variance_ols_safe_: List[str] = field(default_factory=list)
    ohe_dummy_cols_: List[str] = field(default_factory=list)
    multi_binary_cols_: List[str] = field(default_factory=list)

    def fit(self, X_train_raw: pd.DataFrame) -> "DeveloperSalaryPreprocessor":
        X = normalize_object_columns(X_train_raw.copy())

        # Missing flags before imputation. One-hot columns already encode missing as '__Missing__'.
        onehot_set = set(self.onehot_cols)
        missing_rates = X.isna().mean()
        self.missing_indicator_cols_ = [
            col for col, rate in missing_rates.items()
            if rate >= self.missing_flag_threshold and rate > 0 and col not in onehot_set
        ]

        # Numeric medians after numeric engineering.
        X_num = self._engineer_numeric(X)
        self.numeric_medians_ = {
            col: float(X_num[col].median()) if col in X_num.columns and X_num[col].notna().any() else 0.0
            for col in NUMERIC_ALL_COLS
        }

        # Ordinal fallback medians after mapping.
        self.ordinal_fallbacks_ = {}
        for col, mapping in {**self.ordinal_maps, **self.quasi_ordinal_maps}.items():
            if col in X.columns:
                mapped = X[col].replace({"I don't know": np.nan}).map(mapping)
                self.ordinal_fallbacks_[col] = float(mapped.median()) if mapped.notna().any() else 0.0

        # Rare category keep-lists fitted on training only.
        self.rare_keep_values_ = {}
        for col in self.onehot_cols:
            if col not in X.columns:
                continue
            filled = X[col].fillna("__Missing__").astype(str)
            min_count = self.rare_min_count_region if col == "region" else self.rare_min_count_nominal
            counts = filled.value_counts(dropna=False)
            keep = counts[counts >= min_count].index.tolist()
            if not keep and len(counts) > 0:
                keep = [counts.index[0]]
            self.rare_keep_values_[col] = [str(v) for v in keep]

        # Top items for multi-select columns fitted on training only.
        self.multi_top_items_ = {}
        self.multi_unique_item_counts_ = {}
        self.multi_include_count_full_ = {}
        for col, top_n in self.multi_select_top.items():
            if col not in X.columns:
                continue
            counter = Counter()
            for value in X[col]:
                counter.update(parse_multiselect_cell(value))
            top_items = [item for item, _ in counter.most_common(top_n)]
            self.multi_top_items_[col] = top_items
            self.multi_unique_item_counts_[col] = len(counter)
            self.multi_include_count_full_[col] = len(top_items) < len(counter)

        # Build train matrix once to determine zero-variance and final columns.
        X_full_tmp, X_safe_tmp = self.transform(X_train_raw, fit_call=True, remove_zero_variance=False)
        self.zero_variance_full_ = X_full_tmp.columns[X_full_tmp.nunique(dropna=False) <= 1].tolist()
        self.zero_variance_ols_safe_ = X_safe_tmp.columns[X_safe_tmp.nunique(dropna=False) <= 1].tolist()

        X_full_tmp = X_full_tmp.drop(columns=self.zero_variance_full_, errors="ignore")
        X_safe_tmp = X_safe_tmp.drop(columns=self.zero_variance_ols_safe_, errors="ignore")

        self.feature_cols_full_ = X_full_tmp.columns.tolist()
        self.feature_cols_ols_safe_ = X_safe_tmp.columns.tolist()

        self.ohe_dummy_cols_ = [
            c for c in self.feature_cols_full_
            if any(c.startswith(f"{col}__") for col in self.onehot_cols)
        ]
        multi_prefixes = [c.replace(".", "_") + "__" for c in self.multi_top_items_]
        self.multi_binary_cols_ = [c for c in self.feature_cols_full_ if any(c.startswith(p) for p in multi_prefixes)]
        return self

    def transform(
        self,
        X_raw: pd.DataFrame,
        fit_call: bool = False,
        remove_zero_variance: bool = True,
    ) -> Tuple[pd.DataFrame, pd.DataFrame]:
        if not fit_call and not self.feature_cols_full_:
            raise RuntimeError("Preprocessor must be fitted before transform.")

        X = normalize_object_columns(X_raw.copy())
        X = X.drop(columns=[c for c in [ID_COL, TARGET] if c in X.columns])

        pieces_full: List[pd.DataFrame] = []
        pieces_safe: List[pd.DataFrame] = []

        # A. Missingness indicators.
        miss_flags = pd.DataFrame(index=X.index)
        for col in self.missing_indicator_cols_:
            if col in X.columns:
                miss_flags[f"{col}_missing"] = X[col].isna().astype(int)
        if "company.size" in X.columns:
            miss_flags["company.size_unknown"] = (X["company.size"] == "I don't know").astype(int)
        if not miss_flags.empty:
            pieces_full.append(miss_flags)
            pieces_safe.append(miss_flags.copy())

        # B. Numeric features.
        X_num = self._engineer_numeric(X)
        for col, median in self.numeric_medians_.items():
            if col not in X_num.columns:
                X_num[col] = median
            else:
                X_num[col] = X_num[col].fillna(median)
        X_num["coding.years.professional_sq"] = X_num["coding.years.professional"] ** 2
        X_num["experience.years_sq"] = X_num["experience.years"] ** 2
        X_num = X_num.astype(float)
        pieces_full.append(X_num[NUMERIC_FINAL_FULL_COLS].copy())
        pieces_safe.append(X_num[NUMERIC_FINAL_OLS_SAFE_COLS].copy())

        # C. Ordinal and quasi-ordinal features.
        ord_encoded = pd.DataFrame(index=X.index)
        for col, mapping in {**self.ordinal_maps, **self.quasi_ordinal_maps}.items():
            if col in X.columns:
                mapped = X[col].replace({"I don't know": np.nan}).map(mapping)
                ord_encoded[col] = mapped.fillna(self.ordinal_fallbacks_.get(col, 0.0)).astype(float)
        if not ord_encoded.empty:
            pieces_full.append(ord_encoded)
            pieces_safe.append(ord_encoded.copy())

        # D. One-hot features with train-fitted rare grouping.
        ohe_source = pd.DataFrame(index=X.index)
        for col in self.onehot_cols:
            if col not in X.columns:
                continue
            filled = X[col].fillna("__Missing__").astype(str)
            keep = set(self.rare_keep_values_.get(col, []))
            ohe_source[col] = filled.where(filled.isin(keep), "__Rare__")
        if not ohe_source.empty:
            dummies = pd.get_dummies(
                ohe_source,
                columns=ohe_source.columns.tolist(),
                prefix_sep="__",
                drop_first=True,
                dtype=float,
            )
            pieces_full.append(dummies)
            pieces_safe.append(dummies.copy())

        # E. Multi-select features.
        multi_full = pd.DataFrame(index=X.index)
        multi_safe = pd.DataFrame(index=X.index)
        for col, top_items in self.multi_top_items_.items():
            parsed_items = [parse_multiselect_cell(v) for v in X[col]] if col in X.columns else [[] for _ in range(len(X))]
            prefix = col.replace(".", "_")
            counts = [len(items) for items in parsed_items]

            if self.multi_include_count_full_.get(col, True):
                multi_full[f"{prefix}_count"] = counts
            multi_safe[f"{prefix}_count"] = counts

            for item in top_items:
                item_col = f"{prefix}__{sanitize_feature_name(item)}"
                multi_full[item_col] = [int(item in items) for items in parsed_items]

        if not multi_full.empty:
            pieces_full.append(multi_full.astype(float))
        if not multi_safe.empty:
            pieces_safe.append(multi_safe.astype(float))

        X_full = pd.concat(pieces_full, axis=1) if pieces_full else pd.DataFrame(index=X.index)
        X_safe = pd.concat(pieces_safe, axis=1) if pieces_safe else pd.DataFrame(index=X.index)

        X_full = X_full.loc[:, ~X_full.columns.duplicated()].fillna(0).astype(float)
        X_safe = X_safe.loc[:, ~X_safe.columns.duplicated()].fillna(0).astype(float)

        if fit_call:
            if remove_zero_variance:
                X_full = X_full.drop(columns=self.zero_variance_full_, errors="ignore")
                X_safe = X_safe.drop(columns=self.zero_variance_ols_safe_, errors="ignore")
            return X_full, X_safe

        X_full = X_full.drop(columns=self.zero_variance_full_, errors="ignore")
        X_safe = X_safe.drop(columns=self.zero_variance_ols_safe_, errors="ignore")
        X_full = self._align_to_features(X_full, self.feature_cols_full_)
        X_safe = self._align_to_features(X_safe, self.feature_cols_ols_safe_)
        return X_full, X_safe

    @staticmethod
    def _align_to_features(X: pd.DataFrame, feature_cols: List[str]) -> pd.DataFrame:
        X = X.copy()
        for col in feature_cols:
            if col not in X.columns:
                X[col] = 0.0
        extra = [c for c in X.columns if c not in feature_cols]
        if extra:
            X = X.drop(columns=extra)
        return X[feature_cols].fillna(0).astype(float)

    @staticmethod
    def _engineer_numeric(X: pd.DataFrame) -> pd.DataFrame:
        out = pd.DataFrame(index=X.index)
        for col in ["coding.years.professional", "experience.years", "job.satisfaction"]:
            out[col] = pd.to_numeric(X[col], errors="coerce") if col in X.columns else np.nan
        total = pd.to_numeric(X.get("coding.years.total", pd.Series(np.nan, index=X.index)), errors="coerce")
        prof = pd.to_numeric(X.get("coding.years.professional", pd.Series(np.nan, index=X.index)), errors="coerce")
        out["years.before.professional"] = (total - prof).clip(lower=0)
        out["coding.years.professional_sq"] = out["coding.years.professional"] ** 2
        out["experience.years_sq"] = out["experience.years"] ** 2
        return out


# -----------------------------------------------------------------------------
# Build outputs
# -----------------------------------------------------------------------------
def build_feature_engineering_outputs(
    train_path: str = "train.csv",
    test_path: str = "test.csv",
    output_dir: str = "outputs_v6",
    seed: int = DEFAULT_SEED,
    val_size: float = DEFAULT_VAL_SIZE,
    internal_test_size: float = DEFAULT_INTERNAL_TEST_SIZE,
    save_unscaled: bool = False,
) -> Dict:
    """Create and save corrected feature-engineering outputs."""
    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)

    df_train_raw = pd.read_csv(train_path)
    df_kaggle_raw = pd.read_csv(test_path)
    require_columns(df_train_raw, [TARGET], "train")
    require_columns(df_kaggle_raw, [ID_COL], "test")

    print("=" * 78)
    print("ML1 Developer Salary — feature engineering v6")
    print("=" * 78)
    print(f"Train raw : {df_train_raw.shape}")
    print(f"Kaggle raw: {df_kaggle_raw.shape}")

    df_train_raw = normalize_object_columns(df_train_raw)
    df_kaggle_raw = normalize_object_columns(df_kaggle_raw)

    # 1. Internal split. No target cleaning before this split.
    strata_full = (
        make_region_strata(df_train_raw["region"], SPLIT_REGION_MIN_COUNT)
        if SPLIT_STRATIFY_BY_REGION and "region" in df_train_raw.columns
        else None
    )
    df_trainval, df_internal_test = train_test_split(
        df_train_raw,
        test_size=internal_test_size,
        random_state=seed,
        shuffle=True,
        stratify=strata_full,
    )

    strata_trainval = (
        make_region_strata(df_trainval["region"], SPLIT_REGION_MIN_COUNT)
        if SPLIT_STRATIFY_BY_REGION and "region" in df_trainval.columns
        else None
    )
    relative_val_size = val_size / (1.0 - internal_test_size)
    df_train, df_val = train_test_split(
        df_trainval,
        test_size=relative_val_size,
        random_state=seed,
        shuffle=True,
        stratify=strata_trainval,
    )

    print("\nSplit sizes")
    print(f" Train        : {len(df_train):5d} ({len(df_train)/len(df_train_raw):.1%})")
    print(f" Validation   : {len(df_val):5d} ({len(df_val)/len(df_train_raw):.1%})")
    print(f" Internal test: {len(df_internal_test):5d} ({len(df_internal_test)/len(df_train_raw):.1%})")
    print(f" Kaggle test  : {len(df_kaggle_raw):5d}")

    pd.Series(df_train.index, name="original_index").to_csv(output_path / "train_original_indices.csv", index=False)
    pd.Series(df_val.index, name="original_index").to_csv(output_path / "val_original_indices.csv", index=False)
    pd.Series(df_internal_test.index, name="original_index").to_csv(output_path / "internal_test_original_indices.csv", index=False)

    # 2. Target files.
    y_train_raw = df_train[TARGET].astype(float).copy()
    y_val_raw = df_val[TARGET].astype(float).copy()
    y_internal_test_raw = df_internal_test[TARGET].astype(float).copy()

    target_processor = TargetProcessor().fit(df_train)
    y_train_winsor = target_processor.winsorize_training_target(y_train_raw, df_train["region"])

    y_train_floor = y_train_raw.clip(lower=LOW_SALARY_FLOOR_USD)
    y_train_log_raw = safe_log_salary(y_train_raw)
    y_train_log_winsor = safe_log_salary(y_train_winsor)
    y_train_log_floor = safe_log_salary(y_train_floor)
    y_val_log_raw = safe_log_salary(y_val_raw)
    y_internal_test_log_raw = safe_log_salary(y_internal_test_raw)

    train_low_salary_mask = (y_train_raw < LOW_SALARY_FLOOR_USD)
    val_low_salary_mask = (y_val_raw < LOW_SALARY_FLOOR_USD)
    internal_test_low_salary_mask = (y_internal_test_raw < LOW_SALARY_FLOOR_USD)
    train_keep_salary_ge_floor = (~train_low_salary_mask).astype(int)

    low_salary_rows = pd.DataFrame({
        "original_index": df_train.index,
        TARGET: y_train_raw.values,
        "region": df_train["region"].values if "region" in df_train.columns else "",
        "below_floor": train_low_salary_mask.astype(int).values,
    })

    print("\nTarget handling")
    print(" Validation/internal-test targets are NOT cleaned or winsorized.")
    print(" Saved training target variants: raw log, winsorized log, floor-500 log, and drop-mask.")
    print(f" Train raw median        : ${y_train_raw.median():,.0f}")
    print(f" Train winsor median     : ${y_train_winsor.median():,.0f}")
    print(f" Train floor-500 median  : ${y_train_floor.median():,.0f}")
    print(f" Global train winsor bounds: ${target_processor.global_bounds_[0]:,.0f} – ${target_processor.global_bounds_[1]:,.0f}")
    print(f" Rows below ${LOW_SALARY_FLOOR_USD:,.0f}: train={int(train_low_salary_mask.sum())}, val={int(val_low_salary_mask.sum())}, internal_test={int(internal_test_low_salary_mask.sum())}")

    # 3. Feature preprocessing.
    X_train_raw = df_train.drop(columns=[TARGET])
    X_val_raw = df_val.drop(columns=[TARGET])
    X_internal_test_raw = df_internal_test.drop(columns=[TARGET])
    X_kaggle_raw = df_kaggle_raw.copy()

    preprocessor = DeveloperSalaryPreprocessor().fit(X_train_raw)
    X_train_full, X_train_safe = preprocessor.transform(X_train_raw)
    X_val_full, X_val_safe = preprocessor.transform(X_val_raw)
    X_internal_test_full, X_internal_test_safe = preprocessor.transform(X_internal_test_raw)
    X_kaggle_full, X_kaggle_safe = preprocessor.transform(X_kaggle_raw)

    print("\nFeature matrices")
    print(f" FULL train        : {X_train_full.shape}")
    print(f" FULL validation   : {X_val_full.shape}")
    print(f" FULL internal test: {X_internal_test_full.shape}")
    print(f" FULL Kaggle       : {X_kaggle_full.shape}")
    print(f" OLS_SAFE train    : {X_train_safe.shape}")
    print(f" Removed zero-variance FULL columns    : {len(preprocessor.zero_variance_full_)}")
    print(f" Removed zero-variance OLS_SAFE columns: {len(preprocessor.zero_variance_ols_safe_)}")

    # 4. Scaling, fitted only on internal training split.
    scaler_full = StandardScaler()
    scaler_safe = StandardScaler()

    X_train_full_scaled = pd.DataFrame(scaler_full.fit_transform(X_train_full), columns=X_train_full.columns, index=X_train_full.index)
    X_val_full_scaled = pd.DataFrame(scaler_full.transform(X_val_full), columns=X_train_full.columns, index=X_val_full.index)
    X_internal_test_full_scaled = pd.DataFrame(scaler_full.transform(X_internal_test_full), columns=X_train_full.columns, index=X_internal_test_full.index)
    X_kaggle_full_scaled = pd.DataFrame(scaler_full.transform(X_kaggle_full), columns=X_train_full.columns, index=X_kaggle_full.index)

    X_train_safe_scaled = pd.DataFrame(scaler_safe.fit_transform(X_train_safe), columns=X_train_safe.columns, index=X_train_safe.index)
    X_val_safe_scaled = pd.DataFrame(scaler_safe.transform(X_val_safe), columns=X_train_safe.columns, index=X_val_safe.index)
    X_internal_test_safe_scaled = pd.DataFrame(scaler_safe.transform(X_internal_test_safe), columns=X_train_safe.columns, index=X_internal_test_safe.index)
    X_kaggle_safe_scaled = pd.DataFrame(scaler_safe.transform(X_kaggle_safe), columns=X_train_safe.columns, index=X_kaggle_safe.index)

    print("\nScaling")
    print(" StandardScaler fitted on train only.")
    print(f" FULL scaled train mean avg: {X_train_full_scaled.mean().mean():.8f}")
    print(f" FULL scaled train std avg : {X_train_full_scaled.std().mean():.8f}")

    # 5. Save minimal, non-duplicated outputs.
    def save_csv(obj: pd.DataFrame | pd.Series, filename: str, **kwargs) -> None:
        obj.to_csv(output_path / filename, index=False, **kwargs)

    scaled_outputs = {
        "X_train_full_scaled.csv": X_train_full_scaled,
        "X_val_full_scaled.csv": X_val_full_scaled,
        "X_internal_test_full_scaled.csv": X_internal_test_full_scaled,
        "X_kaggle_full_scaled.csv": X_kaggle_full_scaled,
        "X_train_ols_safe_scaled.csv": X_train_safe_scaled,
        "X_val_ols_safe_scaled.csv": X_val_safe_scaled,
        "X_internal_test_ols_safe_scaled.csv": X_internal_test_safe_scaled,
        "X_kaggle_ols_safe_scaled.csv": X_kaggle_safe_scaled,
    }
    for filename, obj in scaled_outputs.items():
        save_csv(obj, filename)

    if save_unscaled:
        unscaled_outputs = {
            "X_train_full.csv": X_train_full,
            "X_val_full.csv": X_val_full,
            "X_internal_test_full.csv": X_internal_test_full,
            "X_kaggle_full.csv": X_kaggle_full,
            "X_train_ols_safe.csv": X_train_safe,
            "X_val_ols_safe.csv": X_val_safe,
            "X_internal_test_ols_safe.csv": X_internal_test_safe,
            "X_kaggle_ols_safe.csv": X_kaggle_safe,
        }
        for filename, obj in unscaled_outputs.items():
            save_csv(obj, filename)

    # Raw salaries for honest RMSE.
    y_train_raw.rename(TARGET).to_csv(output_path / "y_train_raw.csv", index=False, header=True)
    y_val_raw.rename(TARGET).to_csv(output_path / "y_val_raw.csv", index=False, header=True)
    y_internal_test_raw.rename(TARGET).to_csv(output_path / "y_internal_test_raw.csv", index=False, header=True)

    # Log targets and target variants for modeling experiments.
    y_train_log_raw.rename("log_annual_pay_usd_raw").to_csv(output_path / "y_train_log_raw.csv", index=False, header=True)
    y_val_log_raw.rename("log_annual_pay_usd_raw").to_csv(output_path / "y_val_log_raw.csv", index=False, header=True)
    y_internal_test_log_raw.rename("log_annual_pay_usd_raw").to_csv(output_path / "y_internal_test_log_raw.csv", index=False, header=True)
    y_train_log_winsor.rename("log_annual_pay_usd_winsorized_train_only").to_csv(output_path / "y_train_log_winsorized.csv", index=False, header=True)
    y_train_log_floor.rename("log_annual_pay_usd_floor500_train_only").to_csv(output_path / "y_train_log_floor500.csv", index=False, header=True)
    pd.Series(train_keep_salary_ge_floor.values, name="keep_salary_ge_500").to_csv(output_path / "train_keep_salary_ge_500_mask.csv", index=False, header=True)
    low_salary_rows.to_csv(output_path / "train_low_salary_diagnostics.csv", index=False)

    # Submission support.
    df_kaggle_raw[ID_COL].to_csv(output_path / "kaggle_ids.csv", index=False, header=True)
    pd.Series(X_train_full.columns, name="feature").to_csv(output_path / "feature_names_full.csv", index=False, header=True)
    pd.Series(X_train_safe.columns, name="feature").to_csv(output_path / "feature_names_ols_safe.csv", index=False, header=True)

    # Reproducibility objects.
    pipeline_objects = {
        "preprocessor": preprocessor,
        "target_processor": target_processor,
        "scaler_full": scaler_full,
        "scaler_ols_safe": scaler_safe,
        "feature_cols_full": X_train_full.columns.tolist(),
        "feature_cols_ols_safe": X_train_safe.columns.tolist(),
        "missing_indicator_cols": preprocessor.missing_indicator_cols_,
        "numeric_medians": preprocessor.numeric_medians_,
        "ordinal_fallbacks": preprocessor.ordinal_fallbacks_,
        "rare_keep_values": preprocessor.rare_keep_values_,
        "multi_top_items": preprocessor.multi_top_items_,
        "multi_unique_item_counts": preprocessor.multi_unique_item_counts_,
        "multi_include_count_full": preprocessor.multi_include_count_full_,
        "zero_variance_full": preprocessor.zero_variance_full_,
        "zero_variance_ols_safe": preprocessor.zero_variance_ols_safe_,
        "low_salary_floor_usd": LOW_SALARY_FLOOR_USD,
    }
    with open(output_path / "preprocessing_objects.pkl", "wb") as f:
        pickle.dump(pipeline_objects, f)

    # Integrity checks.
    checks = {
        "full_train_nans": int(X_train_full_scaled.isna().sum().sum()),
        "full_train_duplicate_columns": int(X_train_full_scaled.columns.duplicated().sum()),
        "full_train_zero_variance_columns": int((X_train_full_scaled.nunique(dropna=False) <= 1).sum()),
        "ols_safe_train_nans": int(X_train_safe_scaled.isna().sum().sum()),
        "ols_safe_train_duplicate_columns": int(X_train_safe_scaled.columns.duplicated().sum()),
        "ols_safe_train_zero_variance_columns": int((X_train_safe_scaled.nunique(dropna=False) <= 1).sum()),
        "ols_safe_has_all_multiselect_counts": all(
            f"{col.replace('.', '_')}_count" in X_train_safe_scaled.columns
            for col in preprocessor.multi_top_items_.keys()
        ),
        "train_keep_salary_ge_500_mask_length_ok": int(len(train_keep_salary_ge_floor) == len(X_train_full_scaled)),
    }
    checks["full_train_matrix_rank"] = int(np.linalg.matrix_rank(X_train_full_scaled.to_numpy()))
    checks["full_train_n_features"] = int(X_train_full_scaled.shape[1])
    checks["ols_safe_train_matrix_rank"] = int(np.linalg.matrix_rank(X_train_safe_scaled.to_numpy()))
    checks["ols_safe_train_n_features"] = int(X_train_safe_scaled.shape[1])

    summary = {
        "version": "v6",
        "train_raw_shape": list(df_train_raw.shape),
        "kaggle_raw_shape": list(df_kaggle_raw.shape),
        "split_sizes": {
            "train": int(len(df_train)),
            "validation": int(len(df_val)),
            "internal_test": int(len(df_internal_test)),
            "kaggle_test": int(len(df_kaggle_raw)),
        },
        "target_policy": {
            "validation_and_internal_test_targets": "raw_not_cleaned_not_winsorized",
            "available_training_targets": [
                "y_train_log_raw.csv",
                "y_train_log_winsorized.csv",
                "y_train_log_floor500.csv",
                "train_keep_salary_ge_500_mask.csv",
            ],
            "low_salary_floor_usd": LOW_SALARY_FLOOR_USD,
            "low_salary_counts_below_floor": {
                "train": int(train_low_salary_mask.sum()),
                "validation": int(val_low_salary_mask.sum()),
                "internal_test": int(internal_test_low_salary_mask.sum()),
            },
            "global_winsor_bounds_train_only": list(target_processor.global_bounds_),
            "winsor_min_region_n": WINSOR_MIN_REGION_N,
        },
        "feature_policy_changes_v6": {
            "build_vs_buy": "one_hot_not_ordinal",
            "ols_safe_numeric": NUMERIC_FINAL_OLS_SAFE_COLS,
            "full_numeric": NUMERIC_FINAL_FULL_COLS,
            "region_target_encoding": "not_included_here_to_avoid_cv_leakage; implement inside modeling if tested",
            "duan_smearing": "not_a_feature_engineering_step; implement in modeling evaluation",
        },
        "feature_shapes": {
            "full_scaled": list(X_train_full_scaled.shape),
            "ols_safe_scaled": list(X_train_safe_scaled.shape),
        },
        "saved_files_policy": {
            "minimal_outputs": True,
            "unscaled_feature_files_saved": bool(save_unscaled),
            "no_duplicate_alias_files": True,
        },
        "missing_indicator_cols": preprocessor.missing_indicator_cols_,
        "n_missing_indicator_cols": len(preprocessor.missing_indicator_cols_),
        "n_onehot_dummy_cols": len(preprocessor.ohe_dummy_cols_),
        "n_multi_binary_cols": len(preprocessor.multi_binary_cols_),
        "n_multi_select_cols": len(preprocessor.multi_top_items_),
        "multi_count_in_full_cols": [
            col for col, include in preprocessor.multi_include_count_full_.items() if include
        ],
        "multi_count_dropped_from_full_as_redundant": [
            col for col, include in preprocessor.multi_include_count_full_.items() if not include
        ],
        "zero_variance_removed": {
            "full": preprocessor.zero_variance_full_,
            "ols_safe": preprocessor.zero_variance_ols_safe_,
        },
        "checks": checks,
        "multi_top_items": preprocessor.multi_top_items_,
        "rare_keep_values": preprocessor.rare_keep_values_,
    }
    with open(output_path / "feature_engineering_summary.json", "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2, ensure_ascii=False)

    print("\nIntegrity checks")
    for key, value in checks.items():
        print(f" {key}: {value}")

    print("\nSaved outputs")
    print(f" Directory: {output_path.resolve()}")
    print(" Main FULL files: X_train_full_scaled.csv, X_val_full_scaled.csv, X_internal_test_full_scaled.csv, X_kaggle_full_scaled.csv")
    print(" OLS baseline files: X_train_ols_safe_scaled.csv, X_val_ols_safe_scaled.csv, X_internal_test_ols_safe_scaled.csv")
    print(" Target files: y_train_log_raw.csv, y_train_log_winsorized.csv, y_train_log_floor500.csv")
    print(" Correct RMSE files: y_val_raw.csv, y_internal_test_raw.csv")
    print(" Low-salary experiment support: train_keep_salary_ge_500_mask.csv, train_low_salary_diagnostics.csv")
    print("=" * 78)

    return summary


## Run feature engineering

Run this cell from the folder containing `train.csv` and `test.csv`. The outputs are saved to `outputs_v6/`.


In [2]:

summary = build_feature_engineering_outputs(
    train_path="train.csv",
    test_path="test.csv",
    output_dir="outputs_v6",
    save_unscaled=False,
)
summary


ML1 Developer Salary — feature engineering v6
Train raw : (2512, 41)
Kaggle raw: (628, 41)

Split sizes
 Train        :  1758 (70.0%)
 Validation   :   377 (15.0%)
 Internal test:   377 (15.0%)
 Kaggle test  :   628

Target handling
 Validation/internal-test targets are NOT cleaned or winsorized.
 Saved training target variants: raw log, winsorized log, floor-500 log, and drop-mask.
 Train raw median        : $41,854
 Train winsor median     : $41,854
 Train floor-500 median  : $41,854
 Global train winsor bounds: $76 – $178,902
 Rows below $500: train=57, val=13, internal_test=11

Feature matrices
 FULL train        : (1758, 303)
 FULL validation   : (377, 303)
 FULL internal test: (377, 303)
 FULL Kaggle       : (628, 303)
 OLS_SAFE train    : (1758, 108)
 Removed zero-variance FULL columns    : 0
 Removed zero-variance OLS_SAFE columns: 0

Scaling
 StandardScaler fitted on train only.
 FULL scaled train mean avg: -0.00000000
 FULL scaled train std avg : 1.00028454

Integrity checks


{'version': 'v6',
 'train_raw_shape': [2512, 41],
 'kaggle_raw_shape': [628, 41],
 'split_sizes': {'train': 1758,
  'validation': 377,
  'internal_test': 377,
  'kaggle_test': 628},
 'target_policy': {'validation_and_internal_test_targets': 'raw_not_cleaned_not_winsorized',
  'available_training_targets': ['y_train_log_raw.csv',
   'y_train_log_winsorized.csv',
   'y_train_log_floor500.csv',
   'train_keep_salary_ge_500_mask.csv'],
  'low_salary_floor_usd': 500.0,
  'low_salary_counts_below_floor': {'train': 57,
   'validation': 13,
   'internal_test': 11},
  'global_winsor_bounds_train_only': [76.26, 178901.9000000001],
  'winsor_min_region_n': 30},
 'feature_policy_changes_v6': {'build_vs_buy': 'one_hot_not_ordinal',
  'ols_safe_numeric': ['coding.years.professional',
   'job.satisfaction',
   'years.before.professional'],
  'full_numeric': ['coding.years.professional',
   'experience.years',
   'job.satisfaction',
   'years.before.professional',
   'coding.years.professional_sq',
  

## Output check

This confirms that the expected files were created and that the main integrity checks passed.


In [3]:

output_path = Path("outputs_v6")
important_files = [
    "X_train_full_scaled.csv",
    "X_val_full_scaled.csv",
    "X_internal_test_full_scaled.csv",
    "X_kaggle_full_scaled.csv",
    "X_train_ols_safe_scaled.csv",
    "X_val_ols_safe_scaled.csv",
    "X_internal_test_ols_safe_scaled.csv",
    "y_train_log_raw.csv",
    "y_train_log_winsorized.csv",
    "y_train_log_floor500.csv",
    "train_keep_salary_ge_500_mask.csv",
    "y_val_raw.csv",
    "y_internal_test_raw.csv",
    "kaggle_ids.csv",
    "feature_engineering_summary.json",
    "preprocessing_objects.pkl",
]
check = pd.DataFrame({
    "file": important_files,
    "exists": [(output_path / f).exists() for f in important_files],
})
display(check)

with open(output_path / "feature_engineering_summary.json", "r", encoding="utf-8") as f:
    saved_summary = json.load(f)

print("Feature shapes:")
print(json.dumps(saved_summary["feature_shapes"], indent=2))
print("Target policy:")
print(json.dumps(saved_summary["target_policy"], indent=2))
print("v6 feature policy changes:")
print(json.dumps(saved_summary["feature_policy_changes_v6"], indent=2))
print("Checks:")
print(json.dumps(saved_summary["checks"], indent=2))


,file,exists
0,X_train_full_scaled.csv,True
1,X_val_full_scaled.csv,True
2,X_internal_test_full_scaled.csv,True
3,X_kaggle_full_scaled.csv,True
4,X_train_ols_safe_scaled.csv,True
5,X_val_ols_safe_scaled.csv,True
6,X_internal_test_ols_safe_scaled.csv,True
7,y_train_log_raw.csv,True
8,y_train_log_winsorized.csv,True
9,y_train_log_floor500.csv,True


Feature shapes:
{
  "full_scaled": [
    1758,
    303
  ],
  "ols_safe_scaled": [
    1758,
    108
  ]
}
Target policy:
{
  "validation_and_internal_test_targets": "raw_not_cleaned_not_winsorized",
  "available_training_targets": [
    "y_train_log_raw.csv",
    "y_train_log_winsorized.csv",
    "y_train_log_floor500.csv",
    "train_keep_salary_ge_500_mask.csv"
  ],
  "low_salary_floor_usd": 500.0,
  "low_salary_counts_below_floor": {
    "train": 57,
    "validation": 13,
    "internal_test": 11
  },
  "global_winsor_bounds_train_only": [
    76.26,
    178901.9000000001
  ],
  "winsor_min_region_n": 30
}
v6 feature policy changes:
{
  "build_vs_buy": "one_hot_not_ordinal",
  "ols_safe_numeric": [
    "coding.years.professional",
    "job.satisfaction",
    "years.before.professional"
  ],
  "full_numeric": [
    "coding.years.professional",
    "experience.years",
    "job.satisfaction",
    "years.before.professional",
    "coding.years.professional_sq",
    "experience.years_sq"

## Files to use in modeling

For most models, use:

```python
X_train = pd.read_csv("outputs_v6/X_train_full_scaled.csv")
X_val = pd.read_csv("outputs_v6/X_val_full_scaled.csv")
X_internal_test = pd.read_csv("outputs_v6/X_internal_test_full_scaled.csv")
X_kaggle = pd.read_csv("outputs_v6/X_kaggle_full_scaled.csv")
```

For OLS baseline, use:

```python
X_train_ols = pd.read_csv("outputs_v6/X_train_ols_safe_scaled.csv")
X_val_ols = pd.read_csv("outputs_v6/X_val_ols_safe_scaled.csv")
X_internal_test_ols = pd.read_csv("outputs_v6/X_internal_test_ols_safe_scaled.csv")
```

Training target variants:

```python
y_train_log_raw = pd.read_csv("outputs_v6/y_train_log_raw.csv").squeeze()
y_train_log_winsor = pd.read_csv("outputs_v6/y_train_log_winsorized.csv").squeeze()
y_train_log_floor500 = pd.read_csv("outputs_v6/y_train_log_floor500.csv").squeeze()
train_keep_mask = pd.read_csv("outputs_v6/train_keep_salary_ge_500_mask.csv").squeeze().astype(bool)
```

Honest validation RMSE must use raw salary:

```python
y_val_raw = pd.read_csv("outputs_v6/y_val_raw.csv").squeeze()
y_pred_usd = np.exp(y_pred_log)
rmse = np.sqrt(mean_squared_error(y_val_raw, y_pred_usd))
```
